<a href="https://colab.research.google.com/github/DeepthiManthapuram/Building_LLM_Applications/blob/main/TravelBuddy_Agent_using_langchai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [61]:
from google.colab import userdata

In [62]:
gemini_api_key = userdata.get("GEMINI_API_KEY")

INITIALISING MODEL

In [63]:
!pip install langchain-google-genai

In [64]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gemini-3.5-flash-lite",
    model_provider="google_genai",
    api_key=gemini_api_key
)

DESTINATION RESEARCH TOOL

In [65]:
!pip install langchain-tavily

In [66]:
pip install -U langchain-tavily

In [67]:
from langchain_tavily import TavilySearch

tavily_api_key = userdata.get("TAVILY_API_KEY")
destination_research_tool = TavilySearch(
    max_results = 5,
    topic = "general",
    search_depth = "advanced",
    tavily_api_key = tavily_api_key
)

CREATE CUSTOM FLIGHT SEARCH TOOL

In [68]:
from langchain.tools import tool

In [76]:
import requests
from langchain.tools import tool
from google.colab import userdata

rapid_api_key = userdata.get("RAPIDAPI_KEY")


@tool
def search_flights(
    source: str,
    destination: str,
    travel_date: str
) -> dict:
    """
    Search for real flight information between two airports.

    source:
        Origin IATA airport code, e.g. HYD

    destination:
        Destination IATA airport code, e.g. BKK

    travel_date:
        Departure date in YYYY-MM-DD format
    """

    print("\nCalling search_flights tool")
    print(f"Searching flights: {source} -> {destination}")
    print(f"Travel date: {travel_date}")

    url = "https://scraper.run/api/v1/tools/google-flights/search"

    headers = {
        "X-RapidAPI-Key": rapid_api_key,
        "X-RapidAPI-Host": "google-flights-search-and-explore.p.rapidapi.com",
        "Content-Type": "application/json"
    }

    payload = {
        "origin": source,
        "destination": destination,
        "departureDate": travel_date,
        "tripType": "one-way",
        "cabinClass": "economy",
        "currency": "INR",
        "adults": 1,
        "showAllResults": True,
        "includeBookingDetails": True
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload,
        timeout=110
    )

    print("Status code:", response.status_code)

    response.raise_for_status()

    return response.json()

In [71]:
system_prompt = f"""
You are an intelligent Travel Planning Assistant that helps users discover destinations and find suitable flights.

Your job is to understand the user's travel request, gather reliable information using the available tools, and provide a clear, useful, and readable response.

TOOL USAGE:

1. Tavily Search:
   Use Tavily Search to research destination-related information such as:

* Popular destinations
* Tourist attractions
* Best places to visit
* Things to do
* Best time to visit
* Weather and climate information
* Travel recommendations
* Local attractions
* Approximate travel duration between places
* Other useful destination information

2. Flight Search:
Use the flight search tool to search for flights between the source and destination.

Only report flight details that are actually present in the search results.
Do not assume that a flight has available seats unless the source explicitly provides seat availability.

When searching for flights, try to provide:

* Airline name
* Flight number
* Departure airport and airport code
* Arrival airport and airport code
* Departure date
* Departure time
* Arrival date
* Arrival time
* Flight duration
* Number of stops
* Direct or connecting flight
* Layover information
* Ticket price
* Currency
* Seat class
* Available seats, if provided by the search results
* Baggage information, if available
* Terminal information, if available
* Flight status, if available
* Booking link

WORKFLOW:

Follow this process:

Step 1:
Understand the user's request and identify:

* Source location
* Destination
* Travel date, if provided
* Number of travelers, if provided
* Preferred class, if provided
* Any other travel preferences

Step 2:
If the user is asking about destinations or wants destination recommendations, use Tavily Search to research relevant destinations.

Step 3:
Once the destination and source are clear, use the SerpAPI flight search tool to search for flights between the source and destination.

Step 4:
Combine the information from both tools and provide the user with a clear travel response.

IMPORTANT RULES:

* Always use the appropriate tool when current or external information is required.
* Do not invent flight information.
* Do not invent flight prices, timings, seat availability, flight numbers, or booking links.
* If a particular detail is not available from the flight search results, clearly say "Not available in the search results."
* If available seats are not provided, do not assume that seats are available.
* Clearly distinguish between confirmed information and general destination information.
* Prefer current search results when providing flight information.
* If the user has not provided a source or destination, ask for the missing information before searching for flights.
* If the user has not provided a travel date and the flight search requires a date, ask the user for the travel date.
* Do not perform unnecessary searches.
* Use Tavily for destination research and SerpAPI for flight searches. Do not confuse the purpose of the two tools.

RESPONSE FORMAT:

Give the final answer in a clean, readable format.

Use these sections when applicable:

Destination Information

* Destination name
* Why visit
* Popular attractions
* Things to do
* Best time to visit
* Other useful information

Flight Options
For each flight, provide:

Flight 1
Airline:
Flight Number:
Departure:
Arrival:
Departure Time:
Arrival Time:
Duration:
Stops:
Price:
Class:
Available Seats:
Baggage:
Terminal:
Status:
Booking Link:

Flight 2
...

Travel Summary

* Source:
* Destination:
* Travel Date:
* Number of Travelers:
* Important information:

If some information is unavailable, write:
"Not available in the search results."

Do not display raw JSON, tool output, API responses, Python objects, or internal tool information.

Do not mention Tavily, SerpAPI, Gemini, tools, function calls, or internal reasoning in the final response unless the user specifically asks about the implementation.

Your goal is to provide accurate, current, complete, and easy-to-understand travel information based on the information returned by the available tools.

"""

In [75]:
from google.colab import userdata

rapid_api_key = userdata.get("RAPIDAPI_KEY")

print("Key exists:", rapid_api_key is not None)
print("Key length:", len(rapid_api_key) if rapid_api_key else 0)

Key exists: True
Key length: 50


In [77]:
from langchain.agents import create_agent
agent = create_agent(
    model = model,
    tools = [destination_research_tool, search_flights],
    system_prompt = system_prompt,
    debug = True
)

user_query = """
Give me destination recommendations from Hyderabad and find available flights from Hyderabad to those recommended destinations.
Include destination information and complete flight details such as airline, flight number, departure time, arrival time, duration, stops, price, class, available seats, and booking link wherever available.
"""
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": user_query
            }
        ]
    }
)

print(response["messages"][-1].content)

[values] {'messages': [HumanMessage(content='\nGive me destination recommendations from Hyderabad and find available flights from Hyderabad to those recommended destinations.\nInclude destination information and complete flight details such as airline, flight number, departure time, arrival time, duration, stops, price, class, available seats, and booking link wherever available.\n', additional_kwargs={}, response_metadata={}, id='c746002d-1aae-4164-bf4d-f1556ccaf9cc')]}
[updates] {'model': {'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"query": "top tourist destinations from Hyderabad for short vacation direct flights"}'}, '__gemini_function_call_thought_signatures__': {'call_74383': 'EmAKXgFpFH0T/cbI3vjmf1F4A2PqYWQJa+Pu1gGSmqryxeAT3ik9HXp0ZJKkaiqp7ABt+L9RdGVdoU4StGpdu2pm7KI4dB3u9WhrojgTo/6mQtiomNCsrbWg2QIo28bMsI4='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [

HTTPError: 401 Client Error: Unauthorized for url: https://scraper.run/api/v1/tools/google-flights/search